In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import gng_py
import time

import json
import numpy as np
import pandas as pd
import math

In [22]:
import sys,os
sys.path.append("../../helper/")
from gng_aux import processGngModel,getHits,findBMU,detect

# Data preparation

In [14]:
# -------------------------------------------------
# Load & preprocess data
# -------------------------------------------------
iris = load_iris()
X = iris.data
y = iris.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# GNG calculation

In [15]:
ctx = gng_py.PyContext()
ctx.create_system()

ctx.set_parameters(
            input_width = 4,
            weight_rng_min = -1.1,
            weight_rng_max = 1.1,
            edge_removal_age = 50,
            neuron_creation_interval = 200,
            max_train_iterations = 20000,
            target_error = 0.096,
            epsilon_w = 0.1,
            epsilon_n = 0.006,
            alpha = 0.5,
            beta = 0.995,
)

ctx.init_dataset_vec(X_train.flatten())
start = time.time()
ctx.fit()
end = time.time()
model_string = ctx.get_model_string()
print("Time for gng calculation: ",end-start)

Time for gng calculation:  0.42719054222106934


In [16]:
y_train

array([0, 2, 1, 0, 1, 2, 1, 2, 2, 2, 2, 1, 1, 1, 1, 0, 0, 2, 2, 0, 1, 0,
       2, 0, 1, 2, 2, 0, 2, 0, 0, 1, 1, 0, 2, 2, 1, 1, 2, 1, 0, 1, 0, 2,
       0, 0, 2, 0, 0, 0, 0, 1, 2, 1, 0, 2, 1, 2, 0, 2, 0, 1, 2, 0, 1, 1,
       2, 1, 1, 2, 0, 0, 0, 2, 1, 2, 1, 2, 2, 1, 0, 2, 1, 0, 2, 0, 2, 1,
       1, 0, 1, 2, 0, 0, 2, 2, 2, 1, 2, 0, 2, 1, 2, 2, 0, 1, 1, 1, 1, 1,
       0, 2, 1, 1, 0, 0, 0, 0, 1, 0])

In [17]:
df = processGngModel(model_string,y_train)

In [18]:
df = getHits(X_train,y_train,df)

In [19]:

best_neurons_test = findBMU(X_test,df)
best_neurons_train = findBMU(X_train,df)

pred_train = detect(best_neurons_train,y_train,df)
pred_test = detect(best_neurons_test,y_test,df)

In [20]:

# Calculate accuracy
accuracy_train = np.mean(pred_train["predicted_class"] == pred_train["true_class"])
accuracy_test = np.mean(pred_test["predicted_class"] == pred_test["true_class"])
print(f"\nAccuracy Train using GNG: {accuracy_train:.4f}")
print(f"\nAccuracy Test  using GNG: {accuracy_test:.4f}")


Accuracy Train using GNG: 1.0000

Accuracy Test  using GNG: 0.9667
